# 1장 2강: 활성화 함수와 순전파 — 실습문제


## 실습 목표

- 같은 입력에 Sigmoid, Tanh, ReLU를 적용하고 출력의 차이를 설명합니다.
- 비선형 활성화 함수를 넣지 않은 연속된 계산이 하나의 식으로 정리되는 것을 확인합니다.
- 콘크리트 데이터의 두 특성으로 작은 신경망의 순전파를 계산합니다.
- 가중합 결과, 활성화 함수의 출력, 최종 출력, 실제 정답을 구분합니다.

## 실습 환경 / 데이터

- Python, pandas, NumPy를 사용합니다. pandas는 표를 읽는 도구이고 NumPy는 배열과 수식을 계산하는 도구입니다.
- 1장 1강과 같은 **`concrete_data.csv`**를 사용합니다. 노트북과 CSV를 같은 폴더에 두고, 그 폴더를 VS Code에서 열어 실행하세요.
- pandas와 NumPy가 설치된 Python 환경을 노트북의 커널로 선택합니다. 커널은 코드를 실행하는 Python 환경입니다.
- 원본은 1030개 행, 입력 특성 8개와 실제 압축강도 1개로 구성됩니다.
- 손계산을 따라가기 쉽도록 이번 순전파에서는 **시멘트량과 물의 양 두 특성만 선택**합니다. 나머지 특성을 쓸모없다고 판단해 제거한 것이 아닙니다.
- 파일 읽기와 열 이름 공백 제거 코드는 제공합니다. `FileNotFoundError`가 나면 CSV의 위치와 파일명을 먼저 확인하세요.

## 이번 강의의 범위

제공된 **1장 2강 목차: Sigmoid·Tanh·ReLU, 비선형 활성화 함수의 필요성, 순전파**를 기준으로 구성했습니다. 강의 본문은 별도로 제공되지 않았으므로 신경망 크기와 가중치·편향은 이 개념을 설명하기 위해 정한 예시입니다.

가중치와 편향을 고정하고 입력에서 출력까지 계산합니다. 역전파, 미분, 손실함수 계산, 최적화, 모델 학습, 성능 개선은 이번 범위에 포함하지 않습니다. 출력은 **학습 전 예시 모델의 계산 결과**이며 신뢰할 수 있는 압축강도 예측이라고 보지 않습니다.

## 실습 구성

| 구분 | 문제 | 할 일 | 연결되는 내용 |
|---|---|---|---|
| 실습 1 | 1-1 | 같은 값에 Sigmoid·Tanh·ReLU 적용 | 활성화 함수의 출력과 특징 |
| 실습 1 | 1-2 | 활성화 함수가 없는 계산과 ReLU를 넣은 계산 비교 | 비선형 활성화 함수의 필요성 |
| 실습 2 | 2-1 | 첫 샘플로 2 → 2 → 1 신경망 순전파 | 가중합 → 활성화 → 출력 계산 |
| 기초 과제 | 3-1 | 세 번째 샘플에 같은 순전파 적용 | 실습 1·2 복습 |

**실습 2개 + 기초 과제 1개**입니다. 실습 1만 두 세부 문제로 나눴습니다. 과제는 실습에서 사용한 식과 코드를 그대로 적용하는 수준입니다.

## 진행 안내

1. 실습 준비 셀부터 위에서 아래로 실행하세요. 셀은 글이나 코드를 담는 한 칸입니다.
2. 각 문제에서 제공한 변수와 계산 순서를 먼저 읽으세요.
3. 코드 실행 결과와 ‘수식 및 계산 과정’, ‘해석 질문’ 답변을 함께 작성하세요.
4. Sigmoid와 Tanh의 지수 계산은 NumPy에 맡깁니다. 지수 값을 암산하거나 공식을 증명할 필요는 없습니다.
5. 소수는 출력할 때만 반올림합니다. 다음 단계의 계산에는 반올림 전 값을 사용하세요.

## 시작 전에 읽을 용어

| 용어 | 쉽게 풀어쓴 뜻 | 이번 실습에서 보기 |
|---|---|---|
| 가중합과 편향 | 각 입력에 가중치를 곱해 더한 뒤 편향을 추가한 값 | z₁ = X @ W₁ + b₁ |
| 활성화 함수 | 가중합 결과를 다음 단계로 보낼 값으로 변환하는 함수 | Sigmoid, Tanh, ReLU |
| 비선형 | 전체 관계를 하나의 직선 형태로 표현할 수 없음 | ReLU는 0을 경계로 규칙이 달라짐 |
| 순전파 | 정해진 가중치·편향으로 입력에서 출력 방향으로 계산 | 입력 → 은닉층 → 출력층 |
| X | 이번에 선택한 입력값 | 시멘트량, 물의 양 |
| W, b | 가중치와 편향 | 예시로 정한 고정 숫자 |
| Z1 | 첫 계산층에서 활성화 전의 값 | 은닉 뉴런 두 개의 가중합 결과 |
| H1 | 첫 은닉층의 활성화 후 값 | ReLU(Z1) |
| Z2, y_hat | 출력층의 계산값, 모델의 최종 출력 | 이번에는 출력 활성화를 따로 적용하지 않아 같은 값 |
| y_true | 데이터에 기록된 실제 정답 | 측정된 압축강도 |
| @ | 행렬곱을 하는 Python 기호 | 입력과 각 뉴런의 가중치를 곱해 합산 |

`1`, `2`는 층을 구분하는 번호입니다. `y_hat`은 보통 모델의 예측 출력을 나타내지만, 이 실습의 파라미터는 학습되지 않았다는 점을 함께 기억하세요.


### 1장 1강에서 이번 강의로 연결하기

1장 1강에서는 각 입력에 가중치를 곱하고 모두 더한 뒤 편향을 더하는 계산을 배웠습니다. 이번에는 그 계산값에 **활성화 함수**를 적용하고, 나온 값을 다음 층에 보내 최종 출력까지 구합니다.

함수는 ‘값을 넣으면 정해진 규칙으로 결과를 내는 계산 방법’입니다. `ReLU(z)`는 ‘z에 ReLU라는 규칙을 적용한다’고 읽습니다. ReLU라는 숫자와 z를 곱하라는 뜻이 아닙니다. 같은 z라도 어떤 함수를 적용하느냐에 따라 결과가 달라집니다.

### 수식과 코드의 기호부터 읽기

| 표시 | 읽는 방법 | 이번 강의에서 주의할 점 |
|---|---|---|
| $e^t$ | e의 t제곱 | e는 약 2.718인 고정된 수이며, e × t가 아님 |
| $e^{-t}$ | e의 음수 t제곱 | $1/e^t$와 같음. 값 앞에 마이너스를 붙인 $-e^t$와 다름 |
| $sigma(z)$ | z에 Sigmoid 함수를 적용 | σ는 Sigmoid를 나타내기 위한 기호 |
| $max(0,z)$ | 0과 z 중 큰 값을 선택 | 두 값을 더하거나 평균을 내지 않음 |
| $hat y$ 또는 `y_hat` | 모델이 계산한 최종 출력 | 위의 표시를 ‘햇’이라고 읽으며 실제 정답과 구분 |
| 코드의 `=` | 오른쪽 결과를 왼쪽 이름에 저장 | `H1 = ...`은 계산 결과에 H1이라는 이름을 붙이는 것 |
| 코드의 `*` / `@` | 원소별 곱셈 / 행렬곱 | 이번 층 계산은 대응하는 값들을 곱해 **더하는** `@` 사용 |

기호를 외우는 것이 목표는 아닙니다. 먼저 ‘어떤 값을 받아서, 어떤 규칙을 적용하고, 다음에 무엇을 전달하는지’를 말로 설명한 뒤 코드를 읽으세요.


## 실습 준비 — 제공 코드를 그대로 실행하세요

원본의 `fine_aggregate `에는 열 이름 끝 공백이 있습니다. 아래 코드는 열 이름 양 끝의 공백만 제거하며, 숫자와 행 순서는 바꾸지 않습니다. 이 준비 코드는 직접 작성하는 문제가 아닙니다.


In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("concrete_data.csv")
df.columns = df.columns.str.strip()

print("전체 데이터 크기:", df.shape)
print(df[["cement", "water", "concrete_compressive_strength"]].head(3).to_string(index=False))


---

## 실습 1. 활성화 함수와 비선형의 의미 이해하기


### 문제 1-1. Sigmoid·Tanh·ReLU 출력 비교

#### 문제 설명

활성화 함수는 뉴런의 가중합 결과를 받아 다음 단계로 보낼 값을 계산합니다. 함수별 특징을 쉽게 비교하기 위해, 먼저 작은 공통 입력 `z = [-2, -1, 0, 1, 2]`를 사용합니다. 이 다섯 값은 **함수 관찰용 예시이며 CSV에서 추출한 값은 아닙니다.** 실습 2에서 실제 콘크리트 입력으로 구한 가중합에 적용합니다.

#### 먼저 이해하기 — 세 함수

| 함수 | 계산식 | 쉽게 읽기 | 수학적 출력 범위 |
|---|---|---|---|
| Sigmoid | $\sigma(z)=\frac{1}{1+e^{-z}}$ | e의 -z제곱에 1을 더하고, 그 값으로 1을 나눔 | 0보다 크고 1보다 작음 |
| Tanh | $\tanh(z)=\frac{e^z-e^{-z}}{e^z+e^{-z}}$ | 두 지수 값의 차이를 두 지수 값의 합으로 나눔 | -1보다 크고 1보다 작음 |
| ReLU | $\operatorname{ReLU}(z)=\max(0,z)$ | 0과 z 중 더 큰 값을 선택 | 0 이상, 위쪽 제한 없음 |

`e`는 약 2.718인 수학 상수입니다. `np.exp(t)`는 $e^t$를 계산합니다. `e^{-z}`를 `e × (-z)`로 계산하는 것이 아닙니다. Tanh는 `np.tanh(z)`를 사용하므로 지수 식을 길게 직접 작성하지 않아도 됩니다.

`max(0, z)`는 합이나 평균이 아닙니다. 예를 들어 z=-2이면 0, z=2이면 2를 고릅니다. 배열에서는 `np.maximum(0, z)`가 원소마다 이 규칙을 적용합니다.

#### 읽기 도움말 — 무엇에 함수를 적용하나요?

이번 z는 함수의 성질을 관찰하기 위해 준비한 숫자입니다. 실제 뉴런에서는 입력에 가중치를 곱하고 편향까지 더해 z를 구한 다음 활성화 함수에 넣습니다. 가중치 각각에 Sigmoid를 적용하거나, 시멘트량을 먼저 ReLU로 바꾸는 계산과 구분하세요.

‘활성화’라는 이름이 모든 출력을 반드시 0 또는 1로 결정한다는 뜻은 아닙니다. Sigmoid와 Tanh는 소수 값을 낼 수 있고, ReLU는 양수를 원래 크기 그대로 보낼 수 있습니다. 아래 요구사항에서는 각 함수의 출력값 자체를 확인합니다.

#### 요구사항

1. `z = np.array([-2, -1, 0, 1, 2], dtype=float)`를 준비하세요.
2. `1 / (1 + np.exp(-z))`로 Sigmoid 결과 `sigmoid_values`를 구하세요.
3. `np.tanh(z)`로 Tanh 결과 `tanh_values`를 구하세요.
4. `np.maximum(0, z)`로 ReLU 결과 `relu_values`를 구하세요.
5. z와 세 결과를 출력하세요. 소수는 `np.round(배열, 4)`로 소수점 넷째 자리까지 표시하세요.
6. z=0의 세 출력과 ReLU 전체 결과의 계산 과정을 답변란에 적으세요.

#### 작성 도움말

NumPy 배열에 식을 적용하면 각 원소에 같은 계산을 합니다. 반복문이나 직접 만든 함수는 필요하지 않습니다. `dtype=float`는 소수를 다룰 수 있는 배열로 만들겠다는 뜻입니다.

#### 해석 질문

**Q1.** z=0일 때 세 함수의 결과는 각각 무엇인가요? Sigmoid와 Tanh에서는 $e^0=1$을 이용하세요.

**Q2.** 음수를 입력했을 때 세 함수의 결과는 어떻게 다른가요? z=-2를 예로 설명하세요.

**Q3.** ReLU의 결과가 `[0, 0, 0, 1, 2]`인 이유는 무엇인가요? 양수 2도 1로 줄이나요?

**Q4.** Sigmoid의 값이 0과 1 사이이면 그 자체로 신뢰할 수 있는 구매 확률이나 압축강도인가요?

#### 제출 결과

- z와 세 활성화 함수의 출력
- z=0 대입 과정과 ReLU의 원소별 계산
- Q1~Q4 답변


In [ ]:
# 공통 입력입니다.
z = np.array([-2, -1, 0, 1, 2], dtype=float)

# 아래부터 작성하세요.
# Sigmoid: 1 / (1 + np.exp(-z))
# Tanh: np.tanh(z)
# ReLU: np.maximum(0, z)
# np.round(배열, 4)로 출력하세요.


#### 답변 작성란

**수식 및 계산 과정**

- 작성: 

**Q1. z=0일 때 세 함수의 결과는 각각 무엇인가요? Sigmoid와 Tanh에서는 $e^0=1$을 이용하세요.**

- 답변: 

**Q2. 음수를 입력했을 때 세 함수의 결과는 어떻게 다른가요? z=-2를 예로 설명하세요.**

- 답변: 

**Q3. ReLU의 결과가 `[0, 0, 0, 1, 2]`인 이유는 무엇인가요? 양수 2도 1로 줄이나요?**

- 답변: 

**Q4. Sigmoid의 값이 0과 1 사이이면 그 자체로 신뢰할 수 있는 구매 확률이나 압축강도인가요?**

- 답변:


### 문제 1-2. 비선형 활성화 함수가 필요한 이유 확인

#### 문제 설명

층을 여러 개 연결하는 것과 비선형 계산을 넣는 것은 서로 다른 일입니다. 계산을 쉽게 비교하기 위해 입력 하나를 받는 작은 예시를 사용합니다. 이 문제의 x는 **원리를 관찰하기 위한 숫자**이며, 콘크리트의 물리적 측정값은 아닙니다.

아래 두 경우에서 가중치와 편향은 같고, 중간에 ReLU를 적용하는지만 다릅니다.

| 구분 | 첫 단계 | 중간 처리 | 둘째 단계 |
|---|---|---|---|
| 활성화 없음 | z₁ = 3x + 1 | h₁ = z₁ | out_plain = 2h₁ + 4 |
| ReLU 사용 | z₁ = 3x + 1 | h₁ = max(0, z₁) | out_relu = 2h₁ + 4 |

#### 수식을 말로 읽기

활성화가 없으면 둘째 식에 첫째 식을 그대로 넣습니다.

$$2(3x+1)+4=6x+2+4=6x+6$$

이것은 ‘x에 6을 곱하고 6을 더한다’는 한 번의 계산입니다. 두 단계를 거쳤어도 전체가 하나의 직선 형태로 정리됩니다. 편향이 있는 $ax+b$는 엄밀히 **아핀 함수**라고 하지만, 여기서는 새로운 용어를 외우기보다 ‘하나의 직선 형태’라는 뜻에 집중하세요.

ReLU를 넣으면 첫 단계의 값이 음수일 때 0으로 바뀝니다. 따라서 전체 입력 구간을 같은 직선 하나로 표현할 수 없습니다. **ReLU 자체는 구간마다 직선이지만 전체로는 비선형입니다.**

#### 읽기 도움말 — 직선과 비선형을 쉬운 말로 연결하기

직선 형태의 관계에서는 입력이 같은 간격으로 늘어날 때 출력도 일정한 양만큼 변합니다. 예를 들어 ‘입력이 1 증가할 때마다 출력이 6 증가’하는 규칙을 전체 구간에 똑같이 적용하면 직선이 됩니다. 전체 구간에 이런 직선 하나를 적용할 수 없는 관계를 여기서는 비선형이라고 이해하세요.

ReLU는 음수 구간에서는 0을 내고 양수 구간에서는 값을 그대로 냅니다. 두 구간의 규칙이 달라서 꺾이는 모양이 됩니다. 비선형이 되려면 반드시 둥근 곡선이어야 하는 것은 아닙니다. 이번 비교에서는 입력이나 가중치를 바꾸지 않고 **중간의 활성화 유무만** 달리합니다.

#### 요구사항

1. `x_demo = np.array([-2, -1, 0, 1, 2], dtype=float)`를 준비하세요.
2. `z_demo = 3 * x_demo + 1`을 구하세요.
3. 활성화 없는 두 단계 출력 `out_plain = 2 * z_demo + 4`를 구하세요.
4. 합쳐 놓은 한 단계 출력 `out_merged = 6 * x_demo + 6`을 구하세요.
5. `h_demo = np.maximum(0, z_demo)`와 `out_relu = 2 * h_demo + 4`를 구하세요.
6. 중간값과 모든 출력을 표시하고, `np.allclose(out_plain, out_merged)`로 두 결과가 같은지 확인하세요.

#### 작성 도움말

`np.allclose(a, b)`는 같은 위치의 값들이 작은 계산 오차 범위 안에서 같은지 확인합니다. `True`는 비교 조건을 만족한다는 뜻입니다. 반드시 **3x+1을 먼저 계산한 뒤 그 결과에 ReLU를 적용**하세요.

#### 해석 질문

**Q1.** 활성화가 없을 때 두 단계의 식은 어떤 한 식으로 정리되나요? 배열 비교 결과는 무엇인가요?

**Q2.** x=-2일 때 활성화가 없는 출력과 ReLU를 넣은 출력의 계산 과정을 각각 적으세요.

**Q3.** 다섯 입력을 차례로 볼 때 out_plain과 out_relu는 어떻게 증가하나요? ReLU를 넣으면 왜 하나의 직선으로 정리할 수 없나요?

**Q4.** 층만 여러 개 쌓고 비선형 활성화 함수를 전혀 쓰지 않으면 어떤 한계가 있나요? 비선형 함수를 넣으면 항상 예측 성능이 좋아지나요?

#### 제출 결과

- x_demo, z_demo, h_demo와 세 출력 배열
- 활성화 없는 두 출력의 비교 결과
- x=-2의 손계산 및 Q1~Q4 답변


In [ ]:
x_demo = np.array([-2, -1, 0, 1, 2], dtype=float)

# 아래부터 작성하세요.
# 1. z_demo = 3 * x_demo + 1
# 2. 활성화 없는 두 단계와 합쳐진 한 단계 계산
# 3. z_demo에 ReLU 적용 후 둘째 단계 계산
# 4. 중간값·출력과 np.allclose 비교 결과 출력


#### 답변 작성란

**수식 및 계산 과정**

- 작성: 

**Q1. 활성화가 없을 때 두 단계의 식은 어떤 한 식으로 정리되나요? 배열 비교 결과는 무엇인가요?**

- 답변: 

**Q2. x=-2일 때 활성화가 없는 출력과 ReLU를 넣은 출력의 계산 과정을 각각 적으세요.**

- 답변: 

**Q3. 다섯 입력을 차례로 볼 때 out_plain과 out_relu는 어떻게 증가하나요? ReLU를 넣으면 왜 하나의 직선으로 정리할 수 없나요?**

- 답변: 

**Q4. 층만 여러 개 쌓고 비선형 활성화 함수를 전혀 쓰지 않으면 어떤 한계가 있나요? 비선형 함수를 넣으면 항상 예측 성능이 좋아지나요?**

- 답변:


---

## 실습 2. 콘크리트 데이터로 신경망 순전파 계산하기


### 문제 2-1. 첫 번째 샘플의 입력부터 최종 출력까지 계산

#### 문제 설명

첫 번째 샘플의 시멘트량 **540**, 물의 양 **162**를 사용합니다. 실제 압축강도는 **79.99**입니다. 실제 정답은 출력과 역할을 구분하기 위해 확인하며 **순전파의 입력에 넣지 않습니다.** 가중치와 편향은 아래에 주어진 값으로 고정합니다.

#### 신경망과 제공 파라미터

**입력 2개 → 은닉 뉴런 2개(ReLU) → 출력 뉴런 1개**의 작은 구조입니다. 순전파의 계산 순서를 익히기 위해 한 은닉층만 사용합니다.

입력 순서는 **시멘트량 → 물의 양**입니다. 이 노트북에서는 입력을 행으로 놓고 `X @ W`로 계산합니다. 따라서 W1의 **행은 입력 특성, 열은 은닉 뉴런**에 대응합니다.

| W1의 행 | 은닉 뉴런 1 | 은닉 뉴런 2 |
|---|---:|---:|
| 시멘트량 | 0.01 | -0.01 |
| 물의 양 | 0.00 | 0.02 |
| 편향 b1 | -3.00 | 1.00 |

마지막 줄의 편향은 W1에 넣는 행이 아니라 **별도 배열 b1**입니다. 은닉 뉴런 1에 더할 편향은 -3, 은닉 뉴런 2에 더할 편향은 1입니다.

| 출력층이 받는 값 | 출력 뉴런의 가중치 |
|---|---:|
| 은닉 뉴런 1의 활성화 후 값 | 2.0 |
| 은닉 뉴런 2의 활성화 후 값 | 1.0 |

출력층의 편향 b2는 0.5입니다. 압축강도처럼 연속적인 수치 하나를 내는 구조를 연습하므로 **이번 출력층에는 Sigmoid·Tanh·ReLU를 적용하지 않고 계산값을 그대로 사용**합니다. 은닉층의 ReLU는 그대로 적용합니다.

#### 수식을 말로 읽기

$$Z1 = XW1 + b1$$
$$H1 = \operatorname{ReLU}(Z1)$$
$$Z2 = H1W2 + b2, \qquad \hat y = Z2$$

1. 입력 두 개에 각 은닉 뉴런의 가중치를 곱하고 합한 뒤 해당 편향을 더합니다.
2. 은닉 뉴런별 계산값에 ReLU를 적용합니다.
3. 활성화 후 값 두 개를 출력층 가중치와 곱해 합하고 출력층 편향을 더합니다.

코드에서 행렬곱은 `@`입니다. `*`로 바꾸면 같은 계산이 되지 않습니다. 계산 순서와 배열 크기를 아래처럼 맞춥니다.

| 계산 | 배열 크기 | 의미 |
|---|---|---|
| X @ W1 | (1,2) @ (2,2) → (1,2) | 샘플 1개의 입력 2개 → 은닉 뉴런 2개의 가중합 |
| + b1 | (1,2)에 (2,)를 더함 | 각 은닉 뉴런의 자리에 해당 편향을 더함 |
| ReLU(Z1) | (1,2) → (1,2) | 원소별로 적용하므로 크기 유지 |
| H1 @ W2 | (1,2) @ (2,1) → (1,1) | 은닉값 2개 → 출력 1개 |
| + b2 | (1,1)에 (1,)를 더함 | 출력에 편향 0.5를 더함 |

#### 읽기 도움말 — 다음 층으로 넘기는 값

먼저 `X_one`으로 `Z1`을 만들고, `Z1`에 ReLU를 적용해 `H1`을 만듭니다. 다음 출력층은 `H1`을 입력으로 받습니다. **Z1은 활성화 전, H1은 활성화 후**라는 차이를 기억하세요. 이 둘이 어떤 샘플에서는 같을 수 있어도 역할은 다릅니다.

W1·b1은 은닉층 계산에 사용하는 값, W2·b2는 출력층 계산에 사용하는 값입니다. 뒤의 1과 2는 층을 구분하는 번호입니다. 순전파가 다음 층의 가중치·편향을 새로 만드는 것은 아닙니다. 이미 주어진 가중치·편향으로 계산한 중간 결과를 다음 층에 전달합니다.

#### 요구사항

1. 제공 코드로 첫 샘플을 `X_one`에 (1,2) 형태로 준비하고, 실제 정답 `y_true`와 함께 출력하세요.
2. `Z1 = X_one @ W1 + b1`을 계산하세요.
3. `H1 = np.maximum(0, Z1)`을 계산하세요.
4. `Z2 = H1 @ W2 + b2`, `y_hat = Z2`로 최종 출력을 구하세요.
5. Z1, H1, y_hat의 값과 Shape를 출력하세요.
6. 비교를 위해 **은닉층 활성화만 생략한** `y_no_activation = Z1 @ W2 + b2`를 출력하세요.
7. 두 은닉 뉴런과 출력 뉴런의 손계산을 적으세요.

#### 작성 도움말

`[0]`은 첫 번째 샘플입니다. `.reshape(1, 2)`는 두 숫자를 한 행·두 열로 놓습니다. 값은 바꾸지 않습니다. 제공 파라미터의 행·열 순서를 그대로 유지하세요. 최종 출력에는 Z1이 아니라 **ReLU를 거친 H1**을 넣습니다.

#### 해석 질문

**Q1.** 첫 번째 은닉 뉴런과 두 번째 은닉 뉴런의 활성화 전 값은 각각 얼마인가요? W1의 어느 값끼리 사용하는지 적으세요.

**Q2.** ReLU 이후 어떤 값이 달라지며, H1의 Shape는 무엇인가요?

**Q3.** 최종 출력과 은닉 활성화를 생략한 출력은 각각 얼마인가요? 달라진 이유는 무엇인가요?

**Q4.** 출력층에 Sigmoid를 적용하면 출력 범위는 어떻게 되나요? 이번에 출력층 활성화를 따로 적용하지 않은 이유는 무엇인가요?

**Q5.** 실제 정답 79.99는 순전파의 어느 계산에 들어가나요? 순전파를 실행한 것이 학습을 수행한 것인가요?

#### 제출 결과

- X_one, y_true, Z1, H1, y_hat 및 지정된 Shape 출력
- 은닉 활성화를 생략한 출력
- 뉴런별 손계산과 Q1~Q5 답변


In [ ]:
# 제공 코드: 첫 샘플과 파라미터를 준비합니다.
X_one = df[["cement", "water"]].to_numpy()[0].reshape(1, 2)
y_true = df["concrete_compressive_strength"].to_numpy()[0]

W1 = np.array([[0.01, -0.01],
               [0.00,  0.02]])
b1 = np.array([-3.0, 1.0])
W2 = np.array([[2.0],
               [1.0]])
b2 = np.array([0.5])

# 아래부터 작성하세요.
# 1. Z1 = X_one @ W1 + b1
# 2. ReLU로 H1 계산
# 3. Z2와 y_hat 계산
# 4. 값과 Shape 출력
# 5. Z1을 그대로 출력층에 넣어 비교 결과 계산


#### 답변 작성란

**수식 및 계산 과정**

- 작성: 

**Q1. 첫 번째 은닉 뉴런과 두 번째 은닉 뉴런의 활성화 전 값은 각각 얼마인가요? W1의 어느 값끼리 사용하는지 적으세요.**

- 답변: 

**Q2. ReLU 이후 어떤 값이 달라지며, H1의 Shape는 무엇인가요?**

- 답변: 

**Q3. 최종 출력과 은닉 활성화를 생략한 출력은 각각 얼마인가요? 달라진 이유는 무엇인가요?**

- 답변: 

**Q4. 출력층에 Sigmoid를 적용하면 출력 범위는 어떻게 되나요? 이번에 출력층 활성화를 따로 적용하지 않은 이유는 무엇인가요?**

- 답변: 

**Q5. 실제 정답 79.99는 순전파의 어느 계산에 들어가나요? 순전파를 실행한 것이 학습을 수행한 것인가요?**

- 답변:


---

## 과제. 같은 순전파를 다른 샘플에 적용하기


### 문제 3-1. 세 번째 샘플의 순전파와 활성화 전후 비교

#### 문제 설명

실습 2와 **같은 신경망과 같은 파라미터**를 사용합니다. 이번에는 원본 CSV의 세 번째 샘플인 시멘트량 **332.5**, 물의 양 **228**을 넣습니다. 실제 압축강도는 **40.27**입니다.

| 항목 | 이번 과제의 설정 |
|---|---|
| 입력 순서 | 시멘트량 → 물의 양 |
| W1 | [[0.01, -0.01], [0.00, 0.02]] |
| b1 | [-3.0, 1.0] |
| 은닉 활성화 | ReLU |
| W2 | [[2.0], [1.0]] |
| b2 | [0.5] |
| 출력 활성화 | 따로 적용하지 않음 |

새로운 모델을 설계하거나 성능을 높이는 문제가 아닙니다. 실습 2의 입력만 바꾸고 계산을 다시 따라가세요.

#### 수식을 말로 읽기

`Z1 = X @ W1 + b1`로 은닉층의 가중합을 구하고, `H1 = np.maximum(0, Z1)`로 ReLU를 적용한 뒤, `y_hat = H1 @ W2 + b2`로 출력합니다. 활성화를 생략한 비교값에는 H1 대신 Z1을 넣습니다.

#### 읽기 도움말 — 실습과 무엇이 같고 다른가요?

이번에는 입력 샘플만 바꾸고, 같은 구조·가중치·편향·ReLU를 사용합니다. 따라서 값이 달라지는 이유는 ‘계산 규칙을 바꿨기 때문’이 아니라 ‘그 규칙에 넣은 입력이 달라졌기 때문’입니다. 실제 정답은 해당 샘플의 측정값을 확인하기 위한 것이며 계산식에 넣지 않습니다.

#### 요구사항

1. 제공 코드로 세 번째 샘플의 입력 `X_task`와 실제 정답 `y_task_true`를 가져오고 출력하세요.
2. `Z1_task`, `H1_task`, `y_task_hat`을 순서대로 계산하세요.
3. 각 중간값과 최종 출력의 값·Shape를 출력하세요.
4. 은닉 활성화를 생략한 `y_task_no_activation`을 계산하고 출력하세요.
5. `np.allclose()`로 두 최종 출력을 비교하세요.
6. 은닉 뉴런별 손계산과 출력층 손계산을 적고 Q1~Q5에 답하세요.

#### 작성 도움말

세 번째 샘플의 배열 위치는 `[2]`입니다. `[0]`은 첫 번째, `[1]`은 두 번째입니다. 파라미터는 아래에 다시 제공하므로 이전 코드에서 바뀌었는지 걱정하지 않고 사용할 수 있습니다. 실습 2-1의 계산 순서를 그대로 적용하세요.

#### 해석 질문

**Q1.** 두 은닉 뉴런의 활성화 전 값은 각각 얼마인가요? 곱셈과 편향 덧셈을 구분해 적으세요.

**Q2.** ReLU를 적용한 뒤 두 값은 바뀌나요? 이유는 무엇인가요?

**Q3.** 최종 출력은 얼마이며 은닉 활성화를 생략한 출력과 같은가요? 계산 과정과 비교 결과를 적으세요.

**Q4.** 이 샘플에서 두 출력이 같으면 모든 입력에서 ReLU가 필요 없다고 결론 내릴 수 있나요? 실습 2의 첫 샘플과 비교하세요.

**Q5.** 실제 정답 40.27과 최종 출력은 역할이 어떻게 다른가요? 샘플이 바뀌면서 어떤 값은 바꾸고 어떤 값은 유지했나요?

#### 제출 결과

- 세 번째 입력·정답, Z1_task·H1_task·y_task_hat의 값과 지정된 Shape
- 활성화 생략 출력과 두 결과의 비교
- 손계산 및 Q1~Q5 답변


In [ ]:
# 제공 코드: 세 번째 샘플과 실습과 동일한 파라미터입니다.
X_task = df[["cement", "water"]].to_numpy()[2].reshape(1, 2)
y_task_true = df["concrete_compressive_strength"].to_numpy()[2]

W1 = np.array([[0.01, -0.01],
               [0.00,  0.02]])
b1 = np.array([-3.0, 1.0])
W2 = np.array([[2.0],
               [1.0]])
b2 = np.array([0.5])

# 아래부터 작성하세요.
# 1. Z1_task 계산
# 2. ReLU로 H1_task 계산
# 3. y_task_hat 계산
# 4. y_task_no_activation 계산
# 5. 값·Shape·np.allclose 비교 결과 출력


#### 답변 작성란

**수식 및 계산 과정**

- 작성: 

**Q1. 두 은닉 뉴런의 활성화 전 값은 각각 얼마인가요? 곱셈과 편향 덧셈을 구분해 적으세요.**

- 답변: 

**Q2. ReLU를 적용한 뒤 두 값은 바뀌나요? 이유는 무엇인가요?**

- 답변: 

**Q3. 최종 출력은 얼마이며 은닉 활성화를 생략한 출력과 같은가요? 계산 과정과 비교 결과를 적으세요.**

- 답변: 

**Q4. 이 샘플에서 두 출력이 같으면 모든 입력에서 ReLU가 필요 없다고 결론 내릴 수 있나요? 실습 2의 첫 샘플과 비교하세요.**

- 답변: 

**Q5. 실제 정답 40.27과 최종 출력은 역할이 어떻게 다른가요? 샘플이 바뀌면서 어떤 값은 바꾸고 어떤 값은 유지했나요?**

- 답변:


---

## 실습 마무리

**1. Sigmoid·Tanh·ReLU의 출력 범위를 각각 설명하세요.**

- 답변: 

**2. 비선형 활성화 함수 없이 가중합·편향 층만 쌓으면 어떤 일이 생기나요?**

- 답변: 

**3. Z1과 H1은 무엇이 다른가요?**

- 답변: 

**4. 순전파의 계산 순서를 적으세요.**

- 답변: 

**5. ReLU 이후 값이 그대로라면 코드 오류인가요?**

- 답변: 

**6. 순전파와 학습은 같은 뜻인가요?**

- 답변:
